# Recommendation demo

For one user, compare the implicit top five, explicit top five, and a combined top five. The combined ranking takes the explicit top 1,000 books and sorts them by `explicit_score * implicit_probability`. Books already seen by the user are excluded. The singleton section separately takes one favorite title and ranks recommendations anonymously with `history_mlp`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from bookrec.data import ITEM_COLUMN, USER_COLUMN, load_dataset
from bookrec.explicit.hyperparameters import DEFAULT_HYPERPARAMETERS as EXPLICIT_DEFAULTS
from bookrec.explicit.model import ExplicitRecommenderMLP
from bookrec.implicit.hyperparameters import DEFAULT_HYPERPARAMETERS as IMPLICIT_DEFAULTS
from bookrec.implicit.model import HISTORY_MLP_ARCHITECTURE_VERSION, create_implicit_model

# Change these values to select the user, ID model, and singleton query.
USER_ID = 11676
IMPLICIT_MODEL_NAME = "mlp"  # "mlp" or "neumf"
SINGLETON_BOOK_TITLE = "Julius Caesar (Oxford School Shakespeare)"

TOP_K = 5
MIN_SINGLETON_INTERACTIONS = 20
MIN_RECOMMENDATION_INTERACTIONS = 5
EXPLICIT_POOL_SIZE = 1_000
BATCH_SIZE = 16_384
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ARTIFACTS = Path("artifacts")
print(f"Device: {DEVICE}")

Device: cuda


/home/nuva/Job/Datasentics/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ratings = load_dataset("Ratings.csv")
books = load_dataset("Books.csv")
ratings[ITEM_COLUMN] = ratings[ITEM_COLUMN].astype(str)
books[ITEM_COLUMN] = books[ITEM_COLUMN].astype(str)

metadata = (
    books[[
        ITEM_COLUMN,
        "Book-Title",
        "Book-Author",
        "Year-Of-Publication",
        "Publisher",
        "Image-URL-M",
    ]]
    .drop_duplicates(ITEM_COLUMN)
)
metadata_isbns = set(metadata[ITEM_COLUMN])

implicit_checkpoint = torch.load(
    ARTIFACTS / "implicit" / IMPLICIT_MODEL_NAME / "model_with_mappings.pt",
    map_location="cpu",
    weights_only=False,
)
implicit_users = implicit_checkpoint["user_to_index"]
implicit_items = implicit_checkpoint["item_to_index"]
implicit_hparams = implicit_checkpoint.get("hyperparameters", IMPLICIT_DEFAULTS)
implicit_model = create_implicit_model(
    IMPLICIT_MODEL_NAME,
    len(implicit_users),
    len(implicit_items),
    implicit_hparams,
).to(DEVICE)
implicit_model.load_state_dict(implicit_checkpoint["model_state_dict"])
implicit_model.eval()

explicit_checkpoint = torch.load(
    ARTIFACTS / "explicit" / "mlp" / "model_with_mappings.pt",
    map_location="cpu",
    weights_only=False,
)
explicit_users = explicit_checkpoint["user_to_index"]
explicit_items = explicit_checkpoint["item_to_index"]
explicit_hparams = explicit_checkpoint.get("hyperparameters", EXPLICIT_DEFAULTS)
explicit_model = ExplicitRecommenderMLP(
    len(explicit_users),
    len(explicit_items),
    embedding_dim=explicit_hparams["embedding_dim"],
    hidden_dims=tuple(explicit_hparams["hidden_dims"]),
    dropout=explicit_hparams["dropout"],
).to(DEVICE)
explicit_model.load_state_dict(explicit_checkpoint["model_state_dict"])
explicit_model.eval()

if USER_ID not in implicit_users or USER_ID not in explicit_users:
    raise ValueError(f"User-ID {USER_ID} must be known to both models")

/home/nuva/Job/Datasentics/.venv/lib/python3.13/site-packages/kagglehub/pandas_datasets.py:92: DtypeWarning: Columns (0: Year-Of-Publication) have mixed types. Specify dtype option on import or set low_memory=False.
  result = read_function(


In [3]:
def score_items(model, user_index, item_to_index):
    index_to_isbn = [None] * len(item_to_index)
    for isbn, item_index in item_to_index.items():
        index_to_isbn[item_index] = str(isbn)

    batches = []
    with torch.inference_mode():
        for start in range(0, len(index_to_isbn), BATCH_SIZE):
            stop = min(start + BATCH_SIZE, len(index_to_isbn))
            items = torch.arange(start, stop, device=DEVICE)
            users = torch.full_like(items, user_index)
            batches.append(model(users, items).cpu())

    return pd.DataFrame({
        ITEM_COLUMN: index_to_isbn,
        "score": torch.cat(batches).numpy(),
    })

In [4]:
from scipy.special import expit

seen_isbns = set(
    ratings.loc[ratings[USER_COLUMN] == USER_ID, ITEM_COLUMN]
)

implicit_scores = score_items(
    implicit_model, implicit_users[USER_ID], implicit_items
).rename(columns={"score": "implicit_logit"})
implicit_logits = implicit_scores["implicit_logit"].to_numpy()
implicit_scores["implicit_probability"] = expit(implicit_logits)
implicit_scores = implicit_scores[
    implicit_scores[ITEM_COLUMN].isin(metadata_isbns)
    & ~implicit_scores[ITEM_COLUMN].isin(seen_isbns)
]

explicit_scores = score_items(
    explicit_model, explicit_users[USER_ID], explicit_items
).rename(columns={"score": "explicit_score"})
explicit_scores = explicit_scores[
    explicit_scores[ITEM_COLUMN].isin(metadata_isbns)
    & ~explicit_scores[ITEM_COLUMN].isin(seen_isbns)
]

implicit_top = (
    implicit_scores
    .nlargest(TOP_K, "implicit_probability")
    .merge(metadata, on=ITEM_COLUMN)
)
explicit_top = (
    explicit_scores
    .nlargest(TOP_K, "explicit_score")
    .merge(metadata, on=ITEM_COLUMN)
)

combined_pool = (
    explicit_scores
    .merge(
        implicit_scores[[ITEM_COLUMN, "implicit_probability"]],
        on=ITEM_COLUMN,
    )
    .nlargest(EXPLICIT_POOL_SIZE, "explicit_score")
)
combined_pool["combined_score"] = (
    combined_pool["explicit_score"]
    * combined_pool["implicit_probability"]
)
combined_top = (
    combined_pool
    .nlargest(TOP_K, "combined_score")
    .merge(metadata, on=ITEM_COLUMN)
)

In [5]:
metadata_columns = [
    "Book-Title",
    "Book-Author",
    "Year-Of-Publication",
    "Publisher",
    ITEM_COLUMN,
    "Image-URL-M",
]

print(f"User-ID {USER_ID} — implicit model: {IMPLICIT_MODEL_NAME}")
print("Top 5 — implicit")
display(implicit_top[[*metadata_columns, "implicit_probability"]])

print("Top 5 — explicit")
display(explicit_top[[*metadata_columns, "explicit_score"]])

print("Top 5 — combined")
display(combined_top[[
    *metadata_columns,
    "explicit_score",
    "implicit_probability",
    "combined_score",
]])

User-ID 11676 — implicit model: mlp
Top 5 — implicit


,Book-Title,Book-Author,Year-Of-Publication,Publisher,ISBN,Image-URL-M,implicit_probability
0,"Girl, Interrupted",SUSANNA KAYSEN,1994,Vintage,0679746048,http://images.amazon.com/images/P/0679746048.0...,0.919267
1,Love in the Time of Cholera (Penguin Great Boo...,Gabriel Garcia Marquez,1994,Penguin Books,0140119906,http://images.amazon.com/images/P/0140119906.0...,0.893576
2,Pretend You Don't See Her,Mary Higgins Clark,1998,Pocket,0671867156,http://images.amazon.com/images/P/0671867156.0...,0.893081
3,The Bad Place,Dean R. Koontz,1994,Berkley Publishing Group,0425124347,http://images.amazon.com/images/P/0425124347.0...,0.873337
4,What to Expect When You're Expecting (Revised ...,Arlene Eisenberg,1996,Workman Pub Co,089480829X,http://images.amazon.com/images/P/089480829X.0...,0.872373


Top 5 — explicit


,Book-Title,Book-Author,Year-Of-Publication,Publisher,ISBN,Image-URL-M,explicit_score
0,Silent Spring: Rachel Carson,Rachel Carson,1987,Houghton Mifflin Company,0395453909,http://images.amazon.com/images/P/0395453909.0...,8.814979
1,Ender's Game (Ender Wiggins Saga (Paperback)),Orson Scott Card,1986,Tor Books,0812533550,http://images.amazon.com/images/P/0812533550.0...,8.791443
2,Johnny Got His Gun,Dalton Trumbo,1983,Bantam Books,0553274325,http://images.amazon.com/images/P/0553274325.0...,8.759417
3,King Lear (3rd Series),William Shakespeare,1997,Arden Shakespeare,017443460X,http://images.amazon.com/images/P/017443460X.0...,8.741116
4,The Rough Guide to the Lord of the Rings (Roug...,Paul Simpson,2003,Rough Guides Limited,1843532751,http://images.amazon.com/images/P/1843532751.0...,8.723100


Top 5 — combined


,Book-Title,Book-Author,Year-Of-Publication,Publisher,ISBN,Image-URL-M,explicit_score,implicit_probability,combined_score
0,"The Return of the King (The Lord of the Rings,...",J.R.R. TOLKIEN,1986,Del Rey,0345339738,http://images.amazon.com/images/P/0345339738.0...,8.515399,0.869498,7.404119
1,The Color Purple,Alice Walker,1990,Pocket,0671727796,http://images.amazon.com/images/P/0671727796.0...,8.143293,0.871307,7.095312
2,The Wide Window (A Series of Unfortunate Event...,Lemony Snicket,2000,HarperCollins,0064407683,http://images.amazon.com/images/P/0064407683.0...,8.174129,0.860253,7.031816
3,Lolita (Vintage International),VLADIMIR NABOKOV,1989,Vintage,0679723161,http://images.amazon.com/images/P/0679723161.0...,8.049829,0.856542,6.895017
4,The Lord of the Rings (Movie Art Cover),J.R.R. Tolkien,2001,Houghton Mifflin Company,0618129022,http://images.amazon.com/images/P/0618129022.0...,8.673269,0.790549,6.856641


## Singleton history inference

Use one mapped book as the complete anonymous history and rank every other mapped book with `history_mlp`. If several ISBN editions match the configured title, the edition with the most training interactions is used. Candidates with fewer than five training interactions are excluded. Stored output from an older architecture is not valid; rerun this section after retraining version 3.

In [6]:
history_checkpoint = torch.load(
    ARTIFACTS / "implicit" / "history_mlp" / "model_with_mappings.pt",
    map_location="cpu",
    weights_only=False,
)
if history_checkpoint.get("architecture_version") != HISTORY_MLP_ARCHITECTURE_VERSION:
    raise ValueError("The history_mlp checkpoint is obsolete; retrain it first")
history_users = history_checkpoint["user_to_index"]
history_items = history_checkpoint["item_to_index"]
training_item_counts = history_checkpoint["training_item_counts"]
history_hparams = history_checkpoint.get("hyperparameters", IMPLICIT_DEFAULTS)
history_model = create_implicit_model(
    "history_mlp",
    len(history_users),
    len(history_items),
    history_hparams,
).to(DEVICE)
history_model.load_state_dict(history_checkpoint["model_state_dict"])
history_model.eval()

mapped_metadata = metadata[metadata[ITEM_COLUMN].isin(history_items)].copy()
titles = mapped_metadata["Book-Title"].astype(str)
exact_match = titles.str.casefold() == SINGLETON_BOOK_TITLE.casefold()
if exact_match.any():
    query_candidates = mapped_metadata[exact_match].copy()
else:
    contains_match = titles.str.contains(
        SINGLETON_BOOK_TITLE, case=False, regex=False, na=False
    )
    query_candidates = mapped_metadata[contains_match].copy()

if query_candidates.empty:
    raise ValueError(
        f"No mapped book title matches {SINGLETON_BOOK_TITLE!r}"
    )

query_candidates["training_interactions"] = query_candidates[ITEM_COLUMN].map(
    lambda isbn: int(training_item_counts[history_items[isbn]])
)
query_book = query_candidates.nlargest(1, "training_interactions")
query_isbn = query_book.iloc[0][ITEM_COLUMN]
query_item = history_items[query_isbn]
query_interactions = int(query_book.iloc[0]["training_interactions"])


def score_from_history(
    model, context_items, item_to_index, item_training_counts
):
    index_to_isbn = [None] * len(item_to_index)
    for isbn, item_index in item_to_index.items():
        index_to_isbn[item_index] = str(isbn)

    history_tensor = torch.tensor(
        context_items, dtype=torch.long, device=DEVICE
    )
    history_offset = torch.tensor([0], dtype=torch.long, device=DEVICE)
    batches = []
    with torch.inference_mode():
        for start in range(0, len(index_to_isbn), BATCH_SIZE):
            stop = min(start + BATCH_SIZE, len(index_to_isbn))
            items = torch.arange(start, stop, device=DEVICE).unsqueeze(0)
            users = torch.zeros_like(items)  # Ignored by history_mlp.
            logits = model(
                users=users,
                items=items,
                history_items=history_tensor,
                history_offset=history_offset,
            )
            batches.append(logits.squeeze(0).cpu())

    return pd.DataFrame({
        ITEM_COLUMN: index_to_isbn,
        "singleton_logit": torch.cat(batches).numpy(),
        "training_interactions": item_training_counts.numpy(),
    })


singleton_scores = score_from_history(
    history_model, [query_item], history_items, training_item_counts
)
singleton_scores["singleton_probability"] = expit(
    singleton_scores["singleton_logit"].to_numpy()
)
singleton_scores = singleton_scores[
    singleton_scores[ITEM_COLUMN].isin(metadata_isbns)
    & (singleton_scores[ITEM_COLUMN] != query_isbn)
    & (
        singleton_scores["training_interactions"]
        >= MIN_RECOMMENDATION_INTERACTIONS
    )
]
singleton_top = (
    singleton_scores
    .nlargest(TOP_K, "singleton_probability")
    .merge(metadata, on=ITEM_COLUMN)
)

print(f"Singleton input — {query_interactions:,} training interactions")
display(query_book[[*metadata_columns, "training_interactions"]])
if query_interactions < MIN_SINGLETON_INTERACTIONS:
    print(
        "Warning: this edition has too little collaborative support for "
        "reliable singleton recommendations."
    )
print(f"Top {TOP_K} — history_mlp singleton recommendations")
display(singleton_top[[
    *metadata_columns,
    "training_interactions",
    "singleton_probability",
]])

Singleton input


,Book-Title,Book-Author,Year-Of-Publication,Publisher,ISBN,Image-URL-M
397,Julius Caesar (Oxford School Shakespeare),William Shakespeare,2001,Oxford University Press,0198320264,http://images.amazon.com/images/P/0198320264.0...


Top 5 — history_mlp singleton recommendations


,Book-Title,Book-Author,Year-Of-Publication,Publisher,ISBN,Image-URL-M,singleton_probability
0,Wild Animus,Rich Shapero,2004,Too Far,0971880107,http://images.amazon.com/images/P/0971880107.0...,0.999004
1,The Lovely Bones: A Novel,Alice Sebold,2002,"Little, Brown",0316666343,http://images.amazon.com/images/P/0316666343.0...,0.997726
2,Angels &amp; Demons,Dan Brown,2001,Pocket Star,0671027360,http://images.amazon.com/images/P/0671027360.0...,0.993763
3,Snow Falling on Cedars,David Guterson,1995,Vintage Books USA,067976402X,http://images.amazon.com/images/P/067976402X.0...,0.991214
4,The Da Vinci Code,Dan Brown,2003,Doubleday,0385504209,http://images.amazon.com/images/P/0385504209.0...,0.989483


## Diagnostic rating evaluation

The combined score is designed for ranking, not rating prediction, so its RMSE and MAE are only diagnostic. To avoid leakage, this evaluation uses explicit ratings that occur in both models' test splits.

In [7]:
from bookrec.data import RATING_COLUMN, split_interactions
from bookrec.explicit.metrics import mae, rmse

explicit_ratings = ratings[ratings[RATING_COLUMN] > 0].reset_index(drop=True)
_, _, explicit_test = split_interactions(explicit_ratings, seed=42)
_, _, implicit_test = split_interactions(ratings, seed=42)

# Keep ratings held out from both models and known to both mappings.
evaluation_rows = explicit_test.merge(
    implicit_test[[USER_COLUMN, ITEM_COLUMN]].drop_duplicates(),
    on=[USER_COLUMN, ITEM_COLUMN],
    how="inner",
)
evaluation_rows = evaluation_rows[
    evaluation_rows[USER_COLUMN].isin(implicit_users)
    & evaluation_rows[USER_COLUMN].isin(explicit_users)
    & evaluation_rows[ITEM_COLUMN].isin(implicit_items)
    & evaluation_rows[ITEM_COLUMN].isin(explicit_items)
].reset_index(drop=True)


def predict_pairs(model, interactions, user_to_index, item_to_index):
    predictions = []
    with torch.inference_mode():
        for start in range(0, len(interactions), BATCH_SIZE):
            batch = interactions.iloc[start:start + BATCH_SIZE]
            users = torch.tensor(
                batch[USER_COLUMN].map(user_to_index).to_numpy(),
                dtype=torch.long,
                device=DEVICE,
            )
            items = torch.tensor(
                batch[ITEM_COLUMN].map(item_to_index).to_numpy(),
                dtype=torch.long,
                device=DEVICE,
            )
            predictions.append(model(users, items).cpu())
    return torch.cat(predictions).numpy()


labels = evaluation_rows[RATING_COLUMN].to_numpy()
explicit_predictions = predict_pairs(
    explicit_model, evaluation_rows, explicit_users, explicit_items
)
implicit_logits = predict_pairs(
    implicit_model, evaluation_rows, implicit_users, implicit_items
)
combined_predictions = explicit_predictions * expit(implicit_logits)

rating_metrics = pd.DataFrame(
    [
        {
            "model": "explicit_mlp",
            "rmse": rmse(labels, explicit_predictions),
            "mae": mae(labels, explicit_predictions),
        },
        {
            "model": f"explicit_x_{IMPLICIT_MODEL_NAME}",
            "rmse": rmse(labels, combined_predictions),
            "mae": mae(labels, combined_predictions),
        },
    ]
).set_index("model")

print(f"Common held-out ratings: {len(evaluation_rows):,}")
display(rating_metrics)

Common held-out ratings: 2,239


,rmse,mae
model,,
explicit_mlp,1.611694,1.230503
explicit_x_mlp,4.804429,4.054447
